# Fase 2: Limpieza de Datos (Data Cleaning)

## Objetivos de esta fase:
1. Cargar el dataset original y preparar el entorno de trabajo mediante clonación automatizada del repositorio.
2. Identificar y eliminar dinámicamente las columnas constantes que no aportan variabilidad estadística[cite: 1].
3. Aislar y excluir la columna identificativa `Name` para prevenir la memorización y fugas de datos (*data leakage*)[cite: 1].
4. Validar las dimensiones finales y exportar el dataset limpio (`dataset_cleaned.csv`) para su consumo directo en las siguientes fases del proyecto.

## 1 Cargar el dataset original y preparar el entorno de trabajo mediante clonación automatizada del repositorio.

Automatizamos la carga del entorno clonando el repositorio y leyendo el dataset original desde la carpeta *data/* para garantizar la reproducibilidad completa del proceso de limpieza.

In [10]:
# 1. Clonar el repositorio oficial desde GitHub para asegurar la persistencia en Colab
!git clone https://github.com/EliasHernandezBar/mini-antivirus-ai

# 2. Cambiar al directorio de trabajo del proyecto
import os
os.chdir('mini-antivirus-ai')

# 3. Cargar el dataset original de la Fase 1
import pandas as pd
import numpy as np

df = pd.read_csv('data/dataset_malwares.csv')
print(f"Dataset original cargado con éxito. Dimensiones iniciales: {df.shape}")

Cloning into 'mini-antivirus-ai'...
remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 27 (delta 7), reused 22 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (27/27), 1.60 MiB | 8.72 MiB/s, done.
Resolving deltas: 100% (7/7), done.
Dataset original cargado con éxito. Dimensiones iniciales: (19611, 79)


## 2 Identificar y eliminar dinámicamente las columnas constantes que no aportan variabilidad estadística.

En la fase de EDA detectamos columnas que no aportan variabilidad estadística (tienen un único valor para todos los registros). Eliminarlas reduce la dimensionalidad innecesaria y previene el ruido en los modelos de Machine Learning.

In [11]:
# Detectar dinámicamente aquellas columnas con un único valor único (constantes)
columnas_constantes = [col for col in df.columns if df[col].nunique() <= 1]

print(f"Columnas constantes identificadas para eliminar: {columnas_constantes}")

# Eliminar las columnas constantes del DataFrame
df = df.drop(columns=columnas_constantes)
print(f"Dimensiones tras eliminar columnas constantes: {df.shape}")

Columnas constantes identificadas para eliminar: ['e_magic', 'SectionMaxEntropy', 'SectionMaxRawsize', 'SectionMaxVirtualsize', 'SectionMinPhysical', 'SectionMinVirtual', 'SectionMinPointerData', 'SectionMainChar']
Dimensiones tras eliminar columnas constantes: (19611, 71)


## 3 Aislar y excluir la columna identificativa `Name` para prevenir la memorización y fugas de datos (*data leakage*)

Tal como analizamos en el EDA, la columna *Name* contiene identificadores únicos por fila y el patrón del prefijo *VirusShare_* asociado al malware. Su eliminación es estricta para evitar la memorización directa (data leakage) y forzar al modelo a aprender patrones estructurales reales.

In [12]:
# Excluir la columna 'Name' si está presente en el DataFrame
if 'Name' in df.columns:
    df = df.drop(columns=['Name'])
    print("La columna 'Name' ha sido excluida correctamente del dataset.")
else:
    print("La columna 'Name' ya no se encuentra en el DataFrame.")

# Verificar el resultado final de la limpieza de columnas
print(f"Dimensiones definitivas tras la limpieza: {df.shape}")

La columna 'Name' ha sido excluida correctamente del dataset.
Dimensiones definitivas tras la limpieza: (19611, 70)


## 4 Validar las dimensiones finales y exportar el dataset limpio (`dataset_cleaned.csv`) para su consumo directo en las siguientes fases del proyecto.

Una buena práctica en ingeniería de datos es exportar los resultados intermedios limpios a un nuevo archivo (dataset_cleaned.csv). De esta forma, la siguiente fase de Feature Engineering consumirá directamente datos limpios y estructurados sin repetir código.texto en cursiva

In [13]:
import os
from google.colab import files

# Asegurar que la carpeta data/ existe (por seguridad)
os.makedirs('data', exist_ok=True)

# Guardar el dataset limpio en un nuevo archivo CSV
ruta_salida = 'data/dataset_cleaned.csv'
df.to_csv(ruta_salida, index=False)

# Forzar la descarga directa del archivo CSV a tu equipo local
files.download('data/dataset_cleaned.csv')

print(f"Dataset limpio exportado con éxito a: {ruta_salida}")
print(f"Registros finales: {df.shape[0]} | Variables predictoras + Objetivo: {df.shape[1]}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Dataset limpio exportado con éxito a: data/dataset_cleaned.csv
Registros finales: 19611 | Variables predictoras + Objetivo: 70


## Conclusiones y Próximos Pasos

### Resumen de la Fase de Limpieza:
* **Reducción de dimensionalidad:** Se eliminaron con éxito las columnas constantes que no aportaban variabilidad estadística al modelo.
* **Mitigación de fugas de datos (*Data Leakage*):** Se excluyó de forma estricta la columna `Name` para evitar que los algoritmos de Machine Learning memoricen los identificadores o exploten el sesgo del prefijo `VirusShare_` detectado en el EDA.
* **Consolidación del dataset:** Se obtuvo un dataset limpio y optimizado con 19.611 registros y exactamente 70 variables listas para su uso.
* **Persistencia:** El dataset resultante ha sido exportado e integrado de forma estructurada en la ruta `data/dataset_cleaned.csv`[cite: 1].

### Decisiones pendientes para la Fase 3 (Feature Engineering):
1. Evaluar y aplicar las transformaciones necesarias sobre las variables predictoras del dataset limpio.
2. Analizar el escalado de los datos para asegurar un rango homogéneo antes del modelado.
3. Realizar la división estratificada del conjunto de datos (*Train/Test Split*) para preservar las proporciones de las clases (benignos vs. malware).